[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/childmindresearch/llm_tracker/blob/main/tutorials/embeddings_tutorial.ipynb)

# From coded quotes to a map

**TL;DR.** You give it a table of quotes, each coded into a construct. You get back a map
of the quotes and a few tables that tell you whether your constructs hold up.

**What you get** (saved as PNG and CSV files):

1. A **map**. Each quote is a dot. Each construct has a color.
2. An **overlap table**. It shows which constructs are hard to tell apart.
3. A **review list**. It shows the most typical and the most unusual quote of each construct.
4. A **spreadsheet** with every quote and its place on the map.

**How to use it.**

1. Run the install cell below, once.
2. Change the **Settings** cell. Nothing else needs changing.
3. Run every cell, from top to bottom (*Runtime → Run all* in Colab, *Run → Run All Cells*
   in Jupyter).

You do not need to read the code. The text between the code cells explains each step.

In [ ]:
!pip install sentence-transformers umap-learn matplotlib pandas scipy openpyxl

## ⚙️ Settings — the only cell you change

**TL;DR.** Leave `DATA_FILE = ""` the first time. It runs on example data. Then point it at
your own file.

In [ ]:
# ── YOUR DATA ────────────────────────────────────────────────────────────────
# Your file (.csv or .xlsx). Leave "" to use the example data.
DATA_FILE = ""

# Column names in your file:
TOPIC_COLUMN = "construct"        # the construct (code, category) of each quote
QUOTE_COLUMN = "quote"            # the quote itself
CONFIDENCE_COLUMN = "confidence"  # how sure the coder was. Use "" if you have none

# Keep only quotes with at least this confidence.
# 1 keeps everything. 2 keeps only the clear quotes.
MIN_CONFIDENCE = 1

# ── LANGUAGE OF YOUR QUOTES ──────────────────────────────────────────────────
# "english", or "other" for Spanish, Portuguese, any other language, or a mix.
LANGUAGE = "english"

# ── THE MAP (optional) ───────────────────────────────────────────────────────
TOPICS_TO_COLOR = 20   # the largest constructs get a color; the rest are grey
NEIGHBORS = 15         # low (5) = many small groups; high (50) = the big picture
SPREAD = 0.1           # low = dots packed tight; high (0.5) = dots spread out

# ── WHERE TO SAVE ────────────────────────────────────────────────────────────
OUTPUT_FOLDER = "output"

## Setup

**TL;DR.** Loads the tools. Just run it.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.colors import hsv_to_rgb
from matplotlib.lines import Line2D
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.spatial.distance import squareform
from sentence_transformers import SentenceTransformer
from umap import UMAP

MODELS = {
    "english": "all-MiniLM-L6-v2",
    "other": "paraphrase-multilingual-MiniLM-L12-v2",
}
MODEL = MODELS[LANGUAGE]
SEED = 42  # fixed, so the map comes out the same every time

output = Path(OUTPUT_FOLDER)
output.mkdir(parents=True, exist_ok=True)
print(f"Language model: {MODEL}")
print(f"Files will be saved to: {output.resolve()}")

## 1. The data

**TL;DR.** One row per quote. One column says the construct, one holds the quote.

Your file should look like this:

| construct       | quote                               | confidence |
|-----------------|-------------------------------------|------------|
| Sleep problems  | "has not slept well in weeks"       | 2          |
| Hopelessness    | "says nothing will ever get better" | 2          |
| Alcohol use     | "drinks a bit more on weekends"     | 1          |

The confidence column is optional. 1 = indirect or unclear, 2 = clear. Quotes with no
confidence (for example, human codings) count as clear.

**Coming from llm_tracker?** Save its coding table as a CSV, and point `DATA_FILE` at it.
The column names already match:

```python
format_coding_table(results).to_csv("coding_table.csv", index=False)
```

### The example data

**TL;DR.** The example phrases from the Suicide Risk Lexicon codebook. Public text, no
patient data.

The codebook describes each construct with short third-person phrases, like
"States wanting to die" or "Reports being unable to sleep". Here each phrase is a quote,
coded into its own construct.

We use 15 of the codebook's constructs (about 200 phrases): suicidality, mood, anxiety,
loneliness, sleep and substance use. Fewer constructs make the map easier to read.

You can skip the next cell. It only holds the phrases.

In [ ]:
EXAMPLE_PHRASES = {
    "Passive suicidal ideation": (
        "States wanting to die; States they would be better off dead; Reports wanting to be "
        "dead; States wanting to die; States wishing they were dead; States wishing they "
        "weren't alive; States wishing they would not wake up; States they would be better "
        "off dead"
    ),
    "Active suicidal ideation": (
        "Mentions jumping off a building; Mentions hanging themselves; Mentions electrocuting "
        "themselves; Mentions burning themselves alive; Mentions setting themselves on fire; "
        "States intent to take their own life; Uses the abbreviation 'kms'; Mentions jumping "
        "off the roof; Mentions hanging themselves; Mentions killing themselves; Mentions "
        "jumping off their balcony; States intent to end their own life; Mentions being hit "
        "by a train; States having written a suicide note; Mentions jumping from the roof"
    ),
    "Lethal means": (
        "Mentions slicing their wrist; References a razor; Mentions shooting themselves; "
        "Mentions stabbing themselves; Mentions drinking themselves to death; References "
        "cutting; References a lethal injection; Reports having attempted an overdose; "
        "References a noose; References a firearm; Mentions jumping off a bridge; Mentions "
        "putting a knife to their throat; Mentions setting themselves on fire; Mentions "
        "hanging themselves; Mentions running in front of traffic"
    ),
    "Self-injury": (
        "References NSSI; Mentions burning themselves; Mentions carving their skin; Mentions "
        "cutting themselves; Mentions cutting their wrists; Mentions electrocuting "
        "themselves; Mentions mutilating themselves; Mentions shooting themselves; Mentions "
        "slicing their wrist; Mentions slitting their wrists; Mentions slitting their wrists; "
        "Mentions stabbing themselves; Mentions stabbing themselves; Mentions stabbing "
        "themselves; Mentions wounding themselves"
    ),
    "Hopelessness": (
        "States having no future; States feeling beyond help; States they can't be helped; "
        "States feeling doomed; Describes things as futile; States feeling futureless; "
        "Expresses hopelessness; Reports hopelessness; States things are useless; States "
        "there is no end to their suffering; States having no future; States nothing can "
        "help; States there is no hope"
    ),
    "Perceived burdensomeness": (
        "States that they annoy others; States that they make things worse; States being "
        "annoying; States others would be better without them; States others would be better "
        "off without them; Reports difficulty asking for help; Reports feeling guilty about "
        "others caring for them; Expresses sorrow for existing"
    ),
    "Entrapment": (
        "Expresses a desire to escape; Describes feeling paralyzed; States there is no "
        "escape; States wanting to get out; States there is no exit; States wanting to "
        "escape; Reports craving release; States wanting to escape pain; States they have to "
        "escape; Reports running out of options; States there is no escape; Describes feeling "
        "caged; States wanting out; References feeling snared; References feeling entrapped"
    ),
    "Depressed mood": (
        "States feeling sad all the time; References MDD; Reports anhedonia; Reports having "
        "cried; Reports depression; Expresses despair; Reports feeling down; Reports feeling "
        "down; Reports feeling low; Reports a low mood; Describes feeling melancholic; "
        "Reports melancholy; Expresses sadness; Expresses sorrow"
    ),
    "Anhedonia": (
        "States not enjoying anything; Reports anhedonia; Reports not enjoying things; "
        "Reports not feeling pleasure; Describes feeling emotionally flat; Describes feeling "
        "emotionally numb; Describes feeling joyless; Reports a lack of motivation; Reports a "
        "lack of pleasure; Reports no enthusiasm; States nothing feels good; States nothing "
        "is fun; Describes feeling uninspired"
    ),
    "Anxiety": (
        "References Xanax; References a phobia; Expresses apprehension; Reports worrying; "
        "Reports feeling nervous; References Xanax; Reports being unable to relax; References "
        "diazepam; Describes themselves as phobic; Reports anxiety; Describes feeling "
        "anxious; Reports feeling worried; References Ativan; References agoraphobia; Reports "
        "social anxiety"
    ),
    "Panic": (
        "Expresses fright; Reports hyperventilating; References a mental breakdown; "
        "References a nervous breakdown; Reports panic; Reports panic attacks; Reports panic "
        "dissociation; Expresses terror"
    ),
    "Loneliness and isolation": (
        "States having no one to turn to; States no one will miss them; States that nobody "
        "thinks about them; States that nobody wants them there; Reports having no friends; "
        "States that no one misses them; States feeling alone; States that no one cares; "
        "States having no one; States not having anyone; Reports feeling ignored; States "
        "having nobody; Expresses feeling lonely; States having no one to talk to; States "
        "having no one they can talk to"
    ),
    "Sleep problems": (
        "Reports being unable to sleep; Reports night terrors; Reports not sleeping; "
        "References a sleep disorder; Reports snoring; References using a C-PAP; Reports "
        "trouble sleeping; Reports insomnia; References sleep apnea; References hypersomnia; "
        "Reports being sleep deprived; References narcolepsy; Reports being sleepless; "
        "Reports oversleeping; References a circadian rhythm disorder"
    ),
    "Alcohol use": (
        "Reports an urge to drink; References a brewery; Mentions rehab; References Alateen; "
        "Mentions a bartender; References a keg; References chardonnay; References liquor; "
        "References cocktails; References alcohol; References beer; References Budweiser; "
        "Reports a hangover; Reports feeling hungover; References whisky"
    ),
    "Substance use": (
        "References a sedative; References a barbiturate; References alprazolam; References "
        "Zoloft; References vodka; References codeine; References codone; Mentions vaping; "
        "References THC; References nembutal; References benzodiazepines; References "
        "intoxicants; References lorazepam; References marijuana; References escitalopram"
    ),
}


def example_data() -> pd.DataFrame:
    """The codebook phrases as a coding table: one row per phrase."""
    rows = [
        {"topic": topic, "quote": phrase.strip(), "confidence": 2}
        for topic, phrases in EXAMPLE_PHRASES.items()
        for phrase in phrases.split(";")
    ]
    return pd.DataFrame(rows)


def load_your_data(path: str) -> pd.DataFrame:
    """Read your file and check it has the columns named in Settings."""
    file = Path(path)
    table = pd.read_excel(file) if file.suffix in {".xlsx", ".xls"} else pd.read_csv(file)

    needed = [TOPIC_COLUMN, QUOTE_COLUMN] + ([CONFIDENCE_COLUMN] if CONFIDENCE_COLUMN else [])
    missing = [c for c in needed if c not in table.columns]
    if missing:
        raise ValueError(
            f"These columns are not in your file: {missing}\n"
            f"Your file has: {list(table.columns)}\n"
            "Fix the column names in the Settings cell."
        )

    table = table.rename(columns={TOPIC_COLUMN: "topic", QUOTE_COLUMN: "quote"})
    if CONFIDENCE_COLUMN:
        table = table.rename(columns={CONFIDENCE_COLUMN: "confidence"})
        table["confidence"] = table["confidence"].fillna(2)  # no score: count it as clear
    else:
        table["confidence"] = 2  # no confidence column: count every quote as clear
    return table[["topic", "quote", "confidence"]]


data = load_your_data(DATA_FILE) if DATA_FILE else example_data()
print(f"{len(data)} quotes in {data['topic'].nunique()} constructs")
if not DATA_FILE:
    print("(example data — set DATA_FILE in Settings to use your own)")
data.head()

## 2. Clean up

**TL;DR.** Drop quotes below `MIN_CONFIDENCE`. Merge quotes that are exactly the same.

- **Low confidence.** With `MIN_CONFIDENCE = 2`, only clear quotes stay on the map. The
  default, 1, keeps them all.
- **Repeats.** The same quote often appears many times. We keep it once and count how many
  times it appeared. Repeated quotes show up as bigger dots on the map.

In [ ]:
def clean(data: pd.DataFrame) -> pd.DataFrame:
    """Drop low-confidence rows and merge identical quotes within a construct."""
    kept = data[data["confidence"] >= MIN_CONFIDENCE].copy()
    kept["quote"] = kept["quote"].astype(str).str.strip()
    kept = kept[kept["quote"] != ""]
    return (
        kept.groupby(["topic", "quote"], as_index=False)
        .size()
        .rename(columns={"size": "times_said"})
    )


quotes = clean(data)
dropped = int((data["confidence"] < MIN_CONFIDENCE).sum())
print(f"{len(data)} rows → {len(quotes)} distinct quotes ({dropped} dropped for low confidence)")
quotes.head()

## 3. Turn each quote into numbers

**TL;DR.** A language model reads each quote and gives it a list of numbers (an
*embedding*). Quotes that mean similar things get similar numbers.

Think of it as a fingerprint of the meaning. "Reports being unable to sleep" and "Reports
insomnia" share no words, but their fingerprints are close. That is what lets us put quotes
on a map by meaning, not by wording.

The model must know your language. That is what `LANGUAGE` in Settings is for. The last
section shows what goes wrong if it does not.

The first run downloads the model (about 100 MB). Later runs are fast.

In [ ]:
def to_numbers(texts: list[str]) -> np.ndarray:
    """One list of numbers per quote, all scaled to the same length."""
    vectors = np.asarray(SentenceTransformer(MODEL).encode(texts), dtype=np.float32)
    lengths = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / np.where(lengths == 0, 1.0, lengths)


numbers = to_numbers(quotes["quote"].tolist())
print(f"{numbers.shape[0]} quotes, {numbers.shape[1]} numbers each")

## 4. How typical is each quote?

**TL;DR.** Each construct has a center: its average quote. Quotes close to the center are
typical. Quotes far from it are unusual, and worth a second look.

We score each quote against its own construct only. Some constructs are naturally broader
than others, so comparing across them would not be fair.

In [ ]:
def construct_centers(numbers: np.ndarray, topics: np.ndarray) -> tuple[list[str], np.ndarray]:
    """The average quote of each construct."""
    names = sorted(set(topics))
    centers = np.stack([numbers[topics == n].mean(axis=0) for n in names])
    lengths = np.linalg.norm(centers, axis=1, keepdims=True)
    return names, centers / np.where(lengths == 0, 1.0, lengths)


def zero_to_one(values: np.ndarray) -> np.ndarray:
    """Rescale scores to 0-1 within a construct, ignoring the extremes."""
    low, high = np.percentile(values, [5, 95])
    if high <= low:
        return np.ones_like(values)
    return np.clip((values - low) / (high - low), 0.0, 1.0)


topics = quotes["topic"].to_numpy()
names, centers = construct_centers(numbers, topics)
construct_similarity = np.clip(centers @ centers.T, -1.0, 1.0)

fit_raw = np.zeros(len(quotes))    # how close each quote is to its own center
typicality = np.zeros(len(quotes))  # the same, rescaled 0-1 within the construct
for i, name in enumerate(names):
    rows = topics == name
    fit_raw[rows] = numbers[rows] @ centers[i]
    typicality[rows] = zero_to_one(fit_raw[rows])

print(f"Scored {len(quotes)} quotes in {len(names)} constructs")

## 5. Flatten it onto a page

**TL;DR.** Each quote has hundreds of numbers. A page has two directions. A method called
UMAP places the quotes on the page so that similar quotes end up close together.

**Read the map as "who sits next to whom".** Distances on the page are not exact. Two groups
far apart are not "twice as different" as two groups a bit closer. For real numbers, use the
overlap table in section 8.

In [ ]:
position = UMAP(
    n_neighbors=NEIGHBORS, min_dist=SPREAD, metric="cosine", random_state=SEED
).fit_transform(numbers)

print(f"Placed {position.shape[0]} quotes on the page")

## 6. The results spreadsheet

**TL;DR.** One row per quote: its construct, how typical it is (0 to 1), and its place on
the map. You can open it in Excel.

In [ ]:
results = quotes.assign(
    typicality=typicality.round(3),
    map_x=position[:, 0].round(3),
    map_y=position[:, 1].round(3),
)

results.to_csv(output / "quotes_with_positions.csv", index=False)
print(f"Saved → {output / 'quotes_with_positions.csv'}")
results.head()

## 7. The map

**TL;DR.** Each dot is a quote. The **color** is its construct. **Darker** means more typical.
**Bigger** means the quote was repeated.

A note on colors. With many constructs, some colors look alike. That is only a problem when
two *similar* constructs get similar colors, because they sit on top of each other. So
similar constructs get opposite colors. Similar colors only repeat between constructs that
sit far apart anyway.

In [ ]:
def order_by_similarity(similarity: np.ndarray) -> np.ndarray:
    """Line up the constructs so that similar ones are next to each other."""
    if len(similarity) < 3:
        return np.arange(len(similarity))
    distance = np.clip(1.0 - similarity, 0.0, None)
    np.fill_diagonal(distance, 0.0)
    return leaves_list(linkage(squareform(distance, checks=False), method="average"))


def pick_colors(order: np.ndarray) -> np.ndarray:
    """Walk the line-up, jumping 137.5° around the color wheel at each step."""
    hues = np.empty(len(order))
    for step, construct in enumerate(order):
        hues[construct] = (step * 0.381966) % 1.0  # 0.381966 of a turn = 137.5°
    return hues


def shade(hue: float, typicality: np.ndarray) -> np.ndarray:
    """One color per construct, darker for more typical quotes."""
    hsv = np.empty((len(typicality), 3))
    hsv[:, 0] = hue
    hsv[:, 1] = 0.25 + 0.70 * typicality
    hsv[:, 2] = 1.00 - 0.45 * typicality
    return hsv_to_rgb(hsv)


counts = {n: int(results.loc[topics == n, "times_said"].sum()) for n in names}
largest_first = sorted(names, key=lambda n: -counts[n])
colored, greyed = largest_first[:TOPICS_TO_COLOR], largest_first[TOPICS_TO_COLOR:]

picked = [names.index(n) for n in colored]
hues = dict(zip(colored, pick_colors(
    order_by_similarity(construct_similarity[np.ix_(picked, picked)])
)))

figure, axes = plt.subplots(figsize=(13, 9))
if greyed:
    grey = np.isin(topics, greyed)
    axes.scatter(position[grey, 0], position[grey, 1], s=8, c="#dcdcdc", linewidths=0)

legend = []
for name in colored:
    rows = topics == name
    axes.scatter(
        position[rows, 0], position[rows, 1],
        s=18 + 16 * np.log1p(results.loc[rows, "times_said"].to_numpy()),
        c=shade(hues[name], typicality[rows]),
        linewidths=0,
    )
    legend.append(
        Line2D([], [], marker="o", linestyle="", markersize=7, markeredgecolor="none",
               markerfacecolor=hsv_to_rgb([hues[name], 0.80, 0.80]),
               label=f"{name[:44]}  (n={counts[name]})")
    )

note = f"Grey = constructs outside the largest {TOPICS_TO_COLOR}." if greyed else ""
axes.set_title(
    f"Each dot is a quote\nColor = construct; darker = more typical. {note}",
    fontsize=11,
)
axes.set_xticks([])
axes.set_yticks([])
for side in axes.spines.values():
    side.set_visible(False)
axes.legend(handles=legend, loc="center left", bbox_to_anchor=(1.01, 0.5),
            fontsize=8, frameon=False, title="Constructs", title_fontsize=9)
figure.tight_layout()
figure.savefig(output / "quote_map.png", dpi=200, bbox_inches="tight")
print(f"Saved → {output / 'quote_map.png'}")
plt.show()

## 8. Are the constructs really distinct?

**TL;DR.** Look at the `gap` column. **Negative = the quotes of this construct sound more
like another construct than like their own.**

Trust this table over the map. It uses the full numbers, not the flattened picture.

- `fits_own`: how close the quotes are, on average, to their own center.
- `closest_other`: the other construct whose center is nearest.
- `fits_other`: how close the two centers are.
- `gap` = `fits_own` − `fits_other`.

In [ ]:
def compare_constructs(names, similarity, own_fit, counts) -> pd.DataFrame:
    """For each construct: how tight it is, and which other construct it blends into."""
    others = similarity.copy()
    np.fill_diagonal(others, -np.inf)
    nearest = others.argmax(axis=1)
    return (
        pd.DataFrame({
            "construct": names,
            "quotes": [counts[n] for n in names],
            "fits_own": [round(own_fit[n], 2) for n in names],
            "closest_other": [names[j] for j in nearest],
            "fits_other": [round(float(others[i, j]), 2) for i, j in enumerate(nearest)],
        })
        .assign(gap=lambda d: (d["fits_own"] - d["fits_other"]).round(2))
        .sort_values("gap")
        .reset_index(drop=True)
    )


own_fit = {n: float(fit_raw[topics == n].mean()) for n in names}
overlap = compare_constructs(names, construct_similarity, own_fit, counts)
overlap.to_csv(output / "construct_overlap.csv", index=False)
print(f"Saved → {output / 'construct_overlap.csv'}\n")
overlap

**How to read it.** The least distinct constructs are at the top. A negative gap means
"take a look", not "something is wrong". It can mean two things:

- **The coding is inconsistent.** Those quotes need review.
- **The two constructs really overlap** in the codebook. Many codebooks do this on purpose.
  That is fine, but then do not treat them as independent later.

**In the example data,** the top of the table makes clinical sense. *Lethal means*,
*Self-injury* and *Active suicidal ideation* all name methods ("Mentions slicing their
wrist"), so they blend. So do *Anxiety* and *Panic*, *Alcohol use* and *Substance use*, and
*Depressed mood* and *Anhedonia*. This is overlap built into the codebook, not a coding error.

## 9. The most typical and the most unusual quotes

**TL;DR.** For each construct: its clearest example, and its odd one out. The odd ones are
your review list.

In [ ]:
review = []
for name in overlap["construct"]:
    ranked = results[results["topic"] == name].sort_values("typicality", ascending=False)
    review.append({"construct": name, "kind": "most typical", "quote": ranked.iloc[0]["quote"]})
    review.append({"construct": name, "kind": "most unusual", "quote": ranked.iloc[-1]["quote"]})

review = pd.DataFrame(review)
review.to_csv(output / "typical_and_unusual_quotes.csv", index=False)
print(f"Saved → {output / 'typical_and_unusual_quotes.csv'}\n")
for name in overlap["construct"].head(5):
    typical, unusual = review.loc[review["construct"] == name, "quote"]
    print(name)
    print(f"   most typical → {typical}")
    print(f"   most unusual → {unusual}\n")

## 10. What you have now

In your output folder:

| file                             | what it is                                         |
|----------------------------------|----------------------------------------------------|
| `quote_map.png`                  | the map, ready for a paper or slides               |
| `construct_overlap.csv`          | which constructs blend into which                  |
| `typical_and_unusual_quotes.csv` | the clearest and the oddest quote per construct    |
| `quotes_with_positions.csv`      | every quote, how typical it is, and its map place  |

**To use your own data:** in Settings, set `DATA_FILE`, check the three column names, set
`LANGUAGE`, and run all cells again.

**Confidentiality.** If your quotes come from clinical records, these files contain them word
for word. Keep the output folder with your source data, and do not share it.

## Optional: why the language setting matters

**TL;DR.** The English model does not understand Spanish. With Spanish quotes, set
`LANGUAGE = "other"`. That one line is the only change.

This section is a demonstration. You can skip it. It downloads both models (about 600 MB
the first time).

The test: six Spanish quotes, from two clearly different constructs. For each quote, we ask
the model which of the other five is most alike. A model that understands Spanish should
always pick one from the same construct.

In [ ]:
SPANISH_TEST = [
    ("Sleep problems", "refiere dificultad para conciliar el sueño"),
    ("Sleep problems", "dice que duerme muy pocas horas"),
    ("Sleep problems", "se despierta varias veces durante la noche"),
    ("Work and money problems", "refiere dificultades económicas en el hogar"),
    ("Work and money problems", "dice que no le alcanza para cubrir los gastos"),
    ("Work and money problems", "perdió el trabajo hace unos meses"),
]

test_constructs = [construct for construct, _ in SPANISH_TEST]
scores = []
for language, model_name in MODELS.items():
    vectors = SentenceTransformer(model_name).encode([quote for _, quote in SPANISH_TEST])
    vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
    similarity = vectors @ vectors.T
    np.fill_diagonal(similarity, -np.inf)
    right = sum(
        test_constructs[i] == test_constructs[int(similarity[i].argmax())]
        for i in range(len(SPANISH_TEST))
    )
    scores.append({
        "LANGUAGE setting": language,
        "model": model_name,
        "picked the right construct": f"{right} of {len(SPANISH_TEST)}",
    })

pd.DataFrame(scores)

The English model picks the right construct for only 1 of the 6 Spanish quotes. The
multilingual model gets all 6. On a map, the wrong model shows up as constructs that do not
form clear groups. That is easy to mistake for bad coding.

**Things you do not need to change for Spanish:**

- **Accents and ñ.** Leave them in. Removing them makes results slightly worse.
- **Construct names.** The model only reads the quotes. The names can be in any language.
- **Mixed languages.** `"other"` handles English and Spanish in the same file. It places a
  Spanish quote near an English quote with the same meaning.

If all your quotes are in English, keep `"english"`. It is faster and a bit better on English.